# Baseline: TimeGAN

TimeGAN (Yoon et al., NeurIPS 2019) for sequences of (payload, IAT), with the normalized flow length as a static feature. Flows are truncated or zero-padded to 50 packets for training, and at most 50 packets are generated per flow, as in the fixed-length design of the original model.

Run by `scripts/run_matrix.py` after the proposed model of the same dataset and seed, whose `split.json` and `test.csv` it reuses. The packets are cleaned with the same function as for the proposed model, and the run stops if its test flows differ from `test.csv`.

In [ ]:
SEED         = 0
DATASET      = 'facebook_video'
RAW_PATH_STR = 'data/facebook_video.csv'
OUT_DIR      = 'runs/facebook_video/seed0'
MODEL_KEY    = 'timegan'
TEST_SIZE    = 0.3              # not used: the split is read from split.json

In [ ]:
import itertools
import os
import random
import sys
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader, TensorDataset
from tqdm import tqdm

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()   # run from the repository root
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / 'scripts'))
from data_utils import atomic_write_csv, load_packets, load_split_checked, postprocess_synthetic

ART = Path(OUT_DIR) / 'artifacts' / MODEL_KEY
ART.mkdir(parents=True, exist_ok=True)
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':16:8'
torch.use_deterministic_algorithms(True)
np.random.seed(SEED)
torch.manual_seed(SEED)
random.seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

FEATURE_DIM       = 2      # (payload, IAT) per packet
STATIC_DIM        = 1      # normalized flow length
Z_DIM             = 4      # noise dimension
HIDDEN_DIM        = 24     # GRU units
NUM_LAYERS        = 3      # GRU layers
MAX_SEQ_LEN       = 50     # packets per sequence
BATCH_SIZE        = 32
GAMMA             = 1.0    # weight of the supervised loss
EPOCHS_EMBEDDER   = 50     # phase 1: embedder and recovery
EPOCHS_SUPERVISED = 50     # phase 2: supervisor
EPOCHS_JOINT      = 100    # phase 3: joint adversarial training

In [ ]:
df = load_packets(RAW_PATH_STR)
df['payload_log']   = np.log1p(df['payload_length'])
df['time_diff_log'] = np.log1p(df['time_diff'])
df['flow_len_log']  = np.log1p(df.groupby('flow_id')['flow_id'].transform('size'))
train_flows, test_flows = load_split_checked(df, OUT_DIR)
train_mask = df['flow_id'].isin(train_flows)
scaler_len = StandardScaler().fit(df.loc[train_mask, ['flow_len_log']])
df['len_z'] = scaler_len.transform(df[['flow_len_log']])
scaler_pkt = StandardScaler().fit(df.loc[train_mask, ['payload_log', 'time_diff_log']])
df[['plog_z', 'dt_z']] = scaler_pkt.transform(df[['payload_log', 'time_diff_log']])


def create_timegan_dataset(dataframe, flow_list, max_len):
    # one sequence of max_len packets per flow: truncated, or zero-padded at the end
    X_list, S_list, L_list = [], [], []
    for fid, group in tqdm(dataframe[dataframe['flow_id'].isin(flow_list)].groupby('flow_id'), desc='flows'):
        vals = group[['plog_z', 'dt_z']].values
        seq_len = len(vals)
        if seq_len >= max_len:
            vals = vals[:max_len, :]
        else:
            vals = np.vstack([vals, np.zeros((max_len - seq_len, 2))])
        X_list.append(vals)
        S_list.append([group['len_z'].iloc[0]])
        L_list.append(min(seq_len, max_len))
    return (torch.tensor(np.array(X_list, dtype=np.float32)), torch.tensor(np.array(S_list, dtype=np.float32)),
            torch.tensor(np.array(L_list, dtype=np.int64)))


train_X, train_S, train_L = create_timegan_dataset(df, train_flows, MAX_SEQ_LEN)
train_dl = DataLoader(TensorDataset(train_X, train_S, train_L), batch_size=BATCH_SIZE, shuffle=True, drop_last=False)

In [ ]:
class TimeGAN(nn.Module):
    def __init__(self, feature_dim, static_dim, hidden_dim, num_layers, z_dim):
        super().__init__()
        # embedder X -> H and recovery H -> X
        self.embedder_rnn = nn.GRU(feature_dim + static_dim, hidden_dim, num_layers, batch_first=True)
        self.embedder_out = nn.Linear(hidden_dim, hidden_dim)
        self.recovery_rnn = nn.GRU(hidden_dim, hidden_dim, num_layers, batch_first=True)
        self.recovery_out = nn.Linear(hidden_dim, feature_dim)
        # generator Z -> E and supervisor H_{t-1} -> H_t
        self.generator_rnn = nn.GRU(z_dim + static_dim, hidden_dim, num_layers, batch_first=True)
        self.generator_out = nn.Linear(hidden_dim, hidden_dim)
        self.supervisor_rnn = nn.GRU(hidden_dim, hidden_dim, num_layers - 1 if num_layers > 1 else 1, batch_first=True)
        self.supervisor_out = nn.Linear(hidden_dim, hidden_dim)
        # discriminator H -> real/fake
        self.discriminator_rnn = nn.GRU(hidden_dim + static_dim, hidden_dim, num_layers, batch_first=True, bidirectional=True)
        self.discriminator_out = nn.Linear(hidden_dim * 2, 1)

    def embed(self, x, s):
        s_expanded = s.unsqueeze(1).repeat(1, x.size(1), 1)
        h, _ = self.embedder_rnn(torch.cat([x, s_expanded], dim=2))
        return torch.sigmoid(self.embedder_out(h))

    def recover(self, h):
        h_out, _ = self.recovery_rnn(h)
        return self.recovery_out(h_out)

    def generate(self, z, s):
        s_expanded = s.unsqueeze(1).repeat(1, z.size(1), 1)
        e, _ = self.generator_rnn(torch.cat([z, s_expanded], dim=2))
        return torch.sigmoid(self.generator_out(e))

    def supervise(self, h):
        h_next, _ = self.supervisor_rnn(h)
        return torch.sigmoid(self.supervisor_out(h_next))

    def discriminate(self, h, s):
        s_expanded = s.unsqueeze(1).repeat(1, h.size(1), 1)
        d_out, _ = self.discriminator_rnn(torch.cat([h, s_expanded], dim=2))
        return self.discriminator_out(d_out)


model = TimeGAN(FEATURE_DIM, STATIC_DIM, HIDDEN_DIM, NUM_LAYERS, Z_DIM).to(device)
opt_er = optim.Adam(itertools.chain(model.embedder_rnn.parameters(), model.embedder_out.parameters(),
                                    model.recovery_rnn.parameters(), model.recovery_out.parameters()), lr=1e-3)
opt_gs = optim.Adam(itertools.chain(model.generator_rnn.parameters(), model.generator_out.parameters(),
                                    model.supervisor_rnn.parameters(), model.supervisor_out.parameters()), lr=1e-3)
opt_d = optim.Adam(itertools.chain(model.discriminator_rnn.parameters(), model.discriminator_out.parameters()), lr=1e-3)


def masked_mse(preds, targets, lengths):
    mask = torch.arange(preds.size(1)).expand(len(lengths), preds.size(1)).to(device) < lengths.unsqueeze(1)
    mask = mask.unsqueeze(-1).float()
    return (((preds - targets) ** 2) * mask).sum() / mask.sum()


def masked_bce(preds, targets, lengths):
    bce = nn.BCEWithLogitsLoss(reduction='none')
    mask = torch.arange(preds.size(1)).expand(len(lengths), preds.size(1)).to(device) < lengths.unsqueeze(1)
    mask = mask.unsqueeze(-1).float()
    return (bce(preds, targets) * mask).sum() / mask.sum()


# phase 1: embedder and recovery as an autoencoder
for epoch in range(EPOCHS_EMBEDDER):
    model.train()
    for X, S, L in train_dl:
        X, S, L = X.to(device), S.to(device), L.to(device)
        opt_er.zero_grad()
        loss_er = masked_mse(model.recover(model.embed(X, S)), X, L) * 10
        loss_er.backward()
        opt_er.step()

# phase 2: supervisor, predicting H_t from H_{t-1} (teacher forcing)
for epoch in range(EPOCHS_SUPERVISED):
    model.train()
    for X, S, L in train_dl:
        X, S, L = X.to(device), S.to(device), L.to(device)
        opt_gs.zero_grad()
        with torch.no_grad():
            H_real = model.embed(X, S)
        loss_s = masked_mse(model.supervise(H_real[:, :-1, :]), H_real[:, 1:, :], torch.clamp(L - 1, min=0))
        loss_s.backward()
        opt_gs.step()

# phase 3: joint adversarial training
for epoch in range(EPOCHS_JOINT):
    model.train()
    for X, S, L in train_dl:
        X, S, L = X.to(device), S.to(device), L.to(device)
        batch_size = X.size(0)

        # generator and supervisor
        opt_gs.zero_grad()
        Z = torch.randn(batch_size, MAX_SEQ_LEN, Z_DIM).to(device)
        E_hat = model.generate(Z, S)
        H_hat = model.supervise(E_hat)
        Y_fake = model.discriminate(H_hat, S)
        loss_u = masked_bce(Y_fake, torch.ones_like(Y_fake), L)
        with torch.no_grad():
            H_real = model.embed(X, S)
        H_next_pred = model.supervise(H_real[:, :-1, :])
        L_shifted = torch.clamp(L - 1, min=0)
        loss_s = masked_mse(H_next_pred, H_real[:, 1:, :], L_shifted)
        loss_v1 = torch.abs(H_hat.mean(0) - H_real.mean(0)).mean()      # moment matching
        loss_v2 = torch.abs(H_hat.std(0) - H_real.std(0)).mean()
        loss_g_total = loss_u + GAMMA * loss_s + (loss_v1 + loss_v2)
        loss_g_total.backward()
        opt_gs.step()

        # embedder and recovery
        opt_er.zero_grad()
        H_real = model.embed(X, S)
        loss_er = masked_mse(model.recover(H_real), X, L) * 10
        loss_s_e = masked_mse(model.supervise(H_real[:, :-1, :]), H_real[:, 1:, :], L_shifted)
        loss_e_total = loss_er + 0.1 * loss_s_e
        loss_e_total.backward()
        opt_er.step()

        # discriminator
        opt_d.zero_grad()
        H_real = model.embed(X, S).detach()
        Y_real = model.discriminate(H_real, S)
        loss_d_real = masked_bce(Y_real, torch.ones_like(Y_real), L)
        H_hat = model.supervise(model.generate(Z, S)).detach()
        Y_fake = model.discriminate(H_hat, S)
        loss_d_fake = masked_bce(Y_fake, torch.zeros_like(Y_fake), L)
        loss_d = loss_d_real + loss_d_fake
        loss_d.backward()
        opt_d.step()
print('training finished')

In [ ]:
# generation: one sequence per test flow, cut to the flow length (at most MAX_SEQ_LEN packets)
records = []
model.eval()
for fid, grp in tqdm(df[df['flow_id'].isin(test_flows)].groupby('flow_id'), desc='generating'):
    z_input = torch.randn(1, MAX_SEQ_LEN, Z_DIM).to(device)
    s_input = torch.tensor([[grp['len_z'].iloc[0]]], dtype=torch.float32).to(device)
    with torch.no_grad():
        x_gen = model.recover(model.supervise(model.generate(z_input, s_input)))
    x_gen = x_gen.cpu().numpy().squeeze(0)[:min(len(grp), MAX_SEQ_LEN), :]
    feats = scaler_pkt.inverse_transform(x_gen)
    payloads = np.clip(np.expm1(feats[:, 0]), 0.0, 1500.0)
    dts = np.maximum(np.clip(np.expm1(feats[:, 1]), 0.0, 3600.0), 0.0)
    for i in range(len(x_gen)):
        records.append([fid, round(payloads[i]), float(dts[i])])
synth_df = pd.DataFrame(records, columns=['flow_id', 'payload_length', 'time_diff'])

torch.save(model.state_dict(), ART / 'timegan_weights.pth')
joblib.dump(scaler_pkt, ART / 'scaler_pkt_timegan.pkl')

synth_df = postprocess_synthetic(synth_df)
atomic_write_csv(synth_df, Path(OUT_DIR) / f'{MODEL_KEY}.csv')
print(f'saved {len(synth_df):,} packets in {synth_df.flow_id.nunique()} flows')